# Statistical validation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/fix/audit-m2-m4-aggregation/notebooks/05_statistical_validation.ipynb)

**Answers:** Editor comments 4 and 6
**Estimated runtime:** 2 h quick / 20 h full · **Hardware:** CPU
**Quick mode:** set `QUICK_MODE = True` in the setup cell for a fast smoke test.

Confidence intervals, permutation nulls, feature-selection stability and effect sizes,
computed from the stored per-fold predictions. Nothing is refit here.

**Two permutation nulls, deliberately distinct.** Unrestricted permutation is the
leakage test and must give AUC ~ 0.50. Within-school permutation preserves each
school's class composition, so its expected value is above chance and is *estimated*.
Conflating them produces a false leakage alarm.

---

## Aggregation contract (audit M2)

The previous version of this notebook globbed every `*_preds.parquet` in the
checkpoint directory, concatenated them and grouped by `task` alone — with no
configuration filter, mixing selectors, repeats and plausible values into a single
frame. At full budget that enters each student up to 250 times and produces a
confidence interval with no defensible meaning. (On the current local checkpoint
directory the prediction files happen to share one fingerprint, so the old code
returns a plausible-looking 0.8735; that is luck, not correctness, and it ends the
moment a resumed run writes a second fingerprint — as has already happened to the
fold-result checkpoints.)

`vlpso_xai.evaluation.aggregate` replaces it with an explicit three-level
hierarchy. Only the first level is a pooling operation.

| Level | Unit | Operation | Why |
|---|---|---|---|
| 1 | outer folds within (task, method, pv, repeat) | **concatenate** | The folds partition the sample, so each student appears exactly once. AUC and the school-clustered BCa bootstrap variance are computed here and nowhere else. |
| 2 | repeats within (task, method, pv) | **average the AUCs** | A repeat is a re-partition of the same students, not a new sample. Its spread is partition noise and is reported as a range, never added to the sampling variance. |
| 3 | plausible values within (task, method) | **Rubin's rules** | `U` = mean within-PV sampling variance, `B` = between-PV variance. This is the only step that yields a publishable CI, and the only one that carries the PV measurement error — FMI ≈ 0.56, so it dominates. |
| — | methods | **never combined** | Compared with `contrast_table`, not pooled. |

Selecting the configuration fingerprint is mandatory. If the checkpoint directory
holds more than one, `load_fold_predictions` raises and lists them rather than
guessing.

---


In [ ]:
# --- Environment setup -------------------------------------------------
# Detects Colab, mounts Drive only when in Colab, installs pinned deps.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK_MODE = True   # set False for the full budget

# --- Where the code comes from, and where the artefacts live ------------
#
# These are now two different places, deliberately.
#
# Running git against a Google Drive mount does not work reliably. Drive's FUSE
# layer reports changed stat metadata for files git has already read, so a fetch
# into a shallow clone dies with "fatal: shallow file has changed since we read
# it". A --depth 1 clone is also single-branch, so fetching any other branch
# lands in FETCH_HEAD and never creates refs/remotes/origin/<branch>. Both of
# those were hit in sequence on this notebook.
#
# So: CODE is cloned fresh to Colab's local disk every session -- it is ~2 MB,
# it takes a second, and it makes a stale checkout structurally impossible. Only
# the artefact directories are symlinked back to Drive, because those are what
# must survive the session. None of them is tracked by git, so nothing is
# shadowed by doing this.
BRANCH = "fix/audit-m2-m4-aggregation"
REPO   = "https://github.com/yazanjer/An_Explainable_AI_Education.git"

CODE_DIR  = Path("/content/vlpso-xai-pisa")                                # local disk
DRIVE_DIR = Path("/content/drive/MyDrive/An_Explainable_AI_Education")     # persistent
PERSIST   = ("data", "data_local", "results", "models")


def _git(*args, fatal=True):
    """Run git and surface what it said. Silent git is how a stale checkout
    reports success."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode and fatal:
        raise RuntimeError(
            "git " + " ".join(args) + f" failed ({r.returncode})\n"
            + (r.stderr or r.stdout).strip()
        )
    return r


if IN_COLAB:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    if CODE_DIR.exists():
        shutil.rmtree(CODE_DIR)
    _git("clone", "--depth", "1", "--branch", BRANCH, REPO, str(CODE_DIR))

    for name in PERSIST:
        target = DRIVE_DIR / name
        target.mkdir(parents=True, exist_ok=True)
        link = CODE_DIR / name
        if link.is_symlink():
            link.unlink()
        elif link.exists():
            shutil.rmtree(link)
        link.symlink_to(target, target_is_directory=True)

    PROJECT = CODE_DIR
    head_branch = _git("-C", str(CODE_DIR), "rev-parse", "--abbrev-ref", "HEAD").stdout.strip()
    head_commit = _git("-C", str(CODE_DIR), "rev-parse", "--short", "HEAD").stdout.strip()
    print("code   :", CODE_DIR, f"({head_branch} @ {head_commit})")
    print("drive  :", DRIVE_DIR, "->", ", ".join(PERSIST))
    if head_branch != BRANCH:
        raise RuntimeError(
            f"Expected branch {BRANCH!r} but HEAD is on {head_branch!r}. "
            "Nothing below is trustworthy; stop and fix the checkout."
        )

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(CODE_DIR / "requirements.txt")], check=False)
else:
    PROJECT = Path(os.environ.get("VLPSO_PROJECT_ROOT", Path.cwd().parent))

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from vlpso_xai.config import load_config, set_global_seeds, environment_report
cfg = load_config("quick" if QUICK_MODE else "default")
set_global_seeds(cfg.seed)
cfg.paths.mkdirs()
print("project root:", cfg.paths.root)
print("config:", cfg.config_path.name, "| hash:", cfg.hash()[:12])

try:
    import vlpso_xai.evaluation.aggregate  # noqa: F401
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "vlpso_xai.evaluation.aggregate is missing, so this checkout predates "
        f"the M2 fix. Every cell below assumes it. Check out '{BRANCH}' (or a "
        "branch that has merged it) and re-run."
    ) from exc


## 1. Inventory the checkpoints

Print what is actually on disk **before** computing anything. Five of the seven
bugs found in this rebuild were caught by reading printed intermediate
quantities, not by unit tests.

Expect **many** fingerprints, and that is correct. `stage_nested` calls
`run_nested_cv` once per (task, PV) on that combination's row subset, and the
fingerprint hashes the row signature — so every task and every PV necessarily
gets its own. A full run is 3 tasks x 10 PVs = **30 fingerprints x 25 folds =
750 folds**. What must *not* happen is two fingerprints for the **same**
(task, method, pv) cell.

That is exactly what a leftover quick-mode run produces, so the budget being
reported is declared explicitly below. Declaring it is a specification; picking
"the newest" or "the biggest" would be a heuristic, and heuristics are how a
stale run gets reported as current.


In [ ]:
import pandas as pd
from vlpso_xai.evaluation.aggregate import (
    inventory, load_fold_predictions, aggregation_table, aggregate_task_method,
)

# --- Declare the run being reported -------------------------------------
# NOT read from cfg: quick.yaml specifies a 3-fold budget, so in QUICK_MODE
# that would silently select the smoke-test cells instead of the real run.
OUTER_SPLITS    = 5      # outer folds per repeat
OUTER_REPEATS   = 5      # repeats
CFG_FINGERPRINT = None   # only needed to break a collision the budget cannot

print(f"config {cfg.config_path.name} specifies "
      f"{cfg.section('cv', 'outer_repeats')} x {cfg.section('cv', 'outer_splits')}; "
      f"reporting {OUTER_REPEATS} x {OUTER_SPLITS}")

inv = inventory(cfg.paths.checkpoints)          # never raises
print(f"\n{len(inv)} cells | {int(inv.total_folds.sum())} folds | "
      f"{inv.cfg_fingerprint.nunique()} fingerprints")
print("\nCV shapes present (repeats x folds):")
print(inv.groupby(["n_repeats", "folds_per_repeat"]).size().rename("cells").to_string())
display(inv)

dupes = inv.groupby(["task", "method", "pv"]).size()
if (dupes > 1).any():
    print("\nCells with more than one configuration — the budget filter must "
          "resolve these:")
    display(inv.merge(dupes[dupes > 1].rename("n_cfgs").reset_index(),
                      on=["task", "method", "pv"]))

cset = load_fold_predictions(
    cfg.paths.checkpoints,
    cfg_fingerprint=CFG_FINGERPRINT,
    outer_splits=OUTER_SPLITS,
    outer_repeats=OUTER_REPEATS,
)
print("\nselected:", len(cset.index), "folds |", len(cset.fingerprints), "cells |",
      len(cset.tasks), "task(s) |", len(cset.methods), "method(s)")
display(cset.budget())
EXPECTED_FOLDS = OUTER_SPLITS

## 2. Level 1-3 aggregation with Rubin's rules

**Level 1 computes the metric inside each fold and averages by fold size — it
does not concatenate the folds.** Each outer fold selects its own estimator in
its own inner loop, so the five score vectors of a repeat are not on a common
scale, and a pooled AUC across them compares students scored by different
models.

That is not hypothetical. On this run, `medium_vs_high` had exactly two
plausible values whose 25 folds all chose GradientBoosting — PV5 and PV7 — and
those were exactly the two whose pooled AUC was stable across repeats
(SD 0.0010 and 0.0008). The eight PVs that mixed in 1–8 RandomForest folds
swung by 0.014–0.031 between repeats, while their *per-fold* AUCs barely moved
(per-repeat mean 0.6956–0.6967). The swing was the score scales, not the
discrimination.

Both estimands are reported. `auc` is the fold-averaged one and is what should
be cited; `auc_pooled` and `pooled_minus_fold_mean` are kept as a diagnostic,
and any cell exceeding the tolerance is logged.

`summary` is the reportable frame: one row per (task, method), with the CI that
carries plausible-value uncertainty. `single_pv = True` marks a quick-mode row
where between-imputation variance is undefined — diagnostics only.


In [ ]:
# --- Bootstrap cost ------------------------------------------------------
# One school-clustered BCa interval per (task, method, pv, repeat). With 3
# tasks x 10 PVs x 5 repeats that is 150 of them, and BCa adds a jackknife over
# ~1,084 schools on top of the resamples. Full settings are hours; start with
# FAST_PASS to see the shape of the answer, then re-run with FAST_PASS = False
# for the numbers that go in the paper.
FAST_PASS = True

if FAST_PASS:
    N_RESAMPLES, BOOT_METHOD = 200, "percentile"
    print("FAST_PASS: 200 percentile resamples. Indicative only — NOT reportable.")
else:
    N_RESAMPLES = cfg.section("statistics", "bootstrap", "n_resamples")
    BOOT_METHOD = cfg.section("statistics", "bootstrap", "method")
    print(f"FULL: {N_RESAMPLES} {BOOT_METHOD} resamples. Expect hours.")
ALPHA = cfg.section("statistics", "bootstrap", "alpha")

agg = aggregation_table(
    cset,
    metric="auc",
    n_resamples=N_RESAMPLES,
    bootstrap_method=BOOT_METHOD,
    alpha=ALPHA,
    expected_folds=EXPECTED_FOLDS,
)

summary = agg["summary"].sort_values(["task", "method"]).reset_index(drop=True)
display(summary[[
    "task", "method", "n_pv", "n_repeats", "n_folds_per_repeat", "n_students",
    "n_schools", "estimand", "estimate", "ci_low", "ci_high", "standard_error",
    "within_variance", "between_variance", "fmi", "single_pv",
    "max_abs_pooled_minus_fold_mean", "mean_between_fold_sd",
]])

# Where the two estimands disagree, the pooled figure is measuring score scales.
div = agg["per_repeat"][["task", "pv", "rep", "auc_fold_mean", "auc_pooled",
                         "pooled_minus_fold_mean"]]
worst = div.reindex(div.pooled_minus_fold_mean.abs().sort_values(ascending=False).index)
print("\nLargest pooled-vs-fold-averaged divergences:")
display(worst.head(10).round(4))
print("\nRepeat-to-repeat SD of each estimand, per task "
      "(fold-averaged should be flat; pooled inflates where folds mix models):")
display(agg["per_repeat"].groupby(["task", "pv"])[["auc_fold_mean", "auc_pooled"]]
        .std().groupby("task").mean().round(4))

if summary["single_pv"].any():
    print("\nNOTE: rows with single_pv=True have no between-imputation variance. "
          "Their intervals understate uncertainty and are quick-mode only.")

outdir = cfg.paths.results / "statistics"
outdir.mkdir(parents=True, exist_ok=True)
tag = "fastpass" if FAST_PASS else "full"
for name, frame in agg.items():
    frame.to_csv(outdir / f"aggregate_{name}_{tag}.csv", index=False)
print("written:", sorted(p.name for p in outdir.glob(f"aggregate_*_{tag}.csv")))

### 2a. What the old pooled number would have been

Kept as a demonstration, not a result. The gap between the two intervals is the
size of the error M2 describes: the pooled interval is narrower because it counts
each student once per (method x repeat x PV) cell.


In [ ]:
# --- What the old pooled number would have been --------------------------
# The duplication factor is free to compute and makes the point on its own.
# The pooled bootstrap is not: it runs over every duplicated row, which is
# precisely why it is wrong, so it is opt-in.
import glob

SHOW_POOLED_CI = False        # set True to spend the minutes

_files = glob.glob(str(cfg.paths.checkpoints / "*_preds.parquet"))
_all = pd.concat([pd.read_parquet(p) for p in _files], ignore_index=True)
print(f"{len(_files)} checkpoint files -> {len(_all):,} pooled rows, "
      f"{_all.get('cfg_fingerprint', pd.Series(dtype=object)).nunique()} fingerprint(s), "
      f"{_all.method.nunique()} method(s)")

rows = []
for task, sub in _all.groupby("task"):
    ok = summary[summary.task == task]
    n_correct = int(ok["n_students"].max()) if not ok.empty else None
    rows.append({
        "task": task,
        "rows_pooled": len(sub),
        "students": n_correct,
        "duplication_factor": round(len(sub) / n_correct, 1) if n_correct else None,
        "correct_ci_width": float((ok["ci_high"] - ok["ci_low"]).max()) if not ok.empty else None,
    })
demo = pd.DataFrame(rows)

if SHOW_POOLED_CI:
    from vlpso_xai.evaluation.metrics import cluster_bootstrap_ci
    widths = []
    for task, sub in _all.groupby("task"):
        naive = cluster_bootstrap_ci(
            sub.y_true.to_numpy(), sub.y_score.to_numpy(), sub.group.to_numpy(),
            metric="auc", n_resamples=200, method="percentile",
        )
        widths.append(naive["ci_high"] - naive["ci_low"])
    demo["pooled_ci_width"] = widths

display(demo)
print("Shown for contrast only. Not written to results/.")

## 3. Permutation nulls

Two nulls, never interchangeable. Unrestricted must return ~0.50; within-school is
*estimated*, not asserted, because it preserves each school's class composition.


In [ ]:
from vlpso_xai.evaluation.permutation import (
    assert_permutation_null_is_chance, decompose_performance,
)

N_PERM = cfg.section("statistics", "permutation", "n_permutations")
print(f"permutations: {N_PERM} (config value; do not reduce for a reported number)")
print("Unrestricted null is asserted against 0.50 +/- "
      f"{cfg.section('statistics', 'permutation', 'tolerance')}.")
print("Within-school null is ESTIMATED. Asserting it against 0.50 produces a "
      "false leakage alarm; that has happened and is documented.")

## 4. Feature-selection stability

In [ ]:
from vlpso_xai.evaluation.stability import stability_table, selection_frequency

sel_path = cfg.paths.results / "selection" / "fold_results.parquet"
if sel_path.exists():
    sel = pd.read_parquet(sel_path)
    st = stability_table(sel[["task", "method", "rep", "fold", "selected"]],
                         all_features=sorted({f for s in sel.selected for f in s}))
    display(st)
else:
    sel = None
    print(f"{sel_path} not found — run notebook 03 first. "
          "Stability and contrasts below are skipped rather than faked.")

## 5. Paired contrasts (audit M4)

Two changes from the previous version.

**The magnitude label is now guarded.** A paired *d* over 3 folds has a standard
error near 0.6, so its interval covers "negligible" and "large" at once. Bands are
emitted only when there are at least `MIN_FOLDS_FOR_MAGNITUDE = 10` matched folds
*and* the bootstrap CI for *d* lies inside a single band. Otherwise the cell reads
`indeterminate (J=3 < 10)` or `indeterminate (CI spans negligible-large)`, and that
string is what belongs in the manuscript table.

**Dropped contrasts are logged.** A skipped pair silently shrinks the multiplicity
family and invalidates every surviving `p_adjusted`.


In [ ]:
from vlpso_xai.evaluation.effect_size import contrast_table, MIN_FOLDS_FOR_MAGNITUDE
import logging; logging.basicConfig(level=logging.WARNING)

if sel is not None:
    lf = sel.rename(columns={"fold": "outer_fold", "rep": "repeat"})
    ct = contrast_table(
        lf, reference="vlpso", metric="auc",
        n_train=int(lf.n_train.mean()), n_test=int(lf.n_test.mean()),
        correction=cfg.section("statistics", "multiplicity", "method"),
    )
    if ct.empty:
        print("No matched contrasts. Nothing to report.")
    else:
        display(ct[["task", "method_b", "n_folds", "mean_difference", "ci_low", "ci_high",
                    "cohens_d_paired", "d_ci_low", "d_ci_high", "magnitude",
                    "underpowered", "p_value", "p_adjusted"]])
        if ct["underpowered"].all():
            print(f"\nEVERY contrast is underpowered at J < {MIN_FOLDS_FOR_MAGNITUDE}. "
                  "The selector comparison cannot support a claim in either "
                  "direction. Re-run notebook 03 at full budget with BPSO and "
                  "VLPSO matched (audit M11) before writing this section.")
        (cfg.paths.results / "statistics").mkdir(parents=True, exist_ok=True)
        ct.to_csv(cfg.paths.results / "statistics" / "contrasts.csv", index=False)